# Plan thématique d'une réunion / d'un enseignement, puis découpage de la vidéo

À partir d'une vidéo (fichier, YouTube, enregistrement Teams/Zoom), ce notebook :

1. **transcrit** la parole avec horodatage (faster-whisper) ;
2. **repère les grandes sections thématiques** (de l'ordre de 5 à 10 minutes, plus si la même notion se prolonge) ;
3. **rédige pour chaque section un thème sous forme de phrase compréhensible et une description** (LLM) ;
4. **découpe la vidéo** selon ces sections et produit des **chapitres** (fichier YouTube, vidéo avec chapitres intégrés).

Résultat visé (`plan_thematique.txt`) :

```
I. Thème : Pourquoi rester attaché à l'appel de Dieu change notre manière de vivre au quotidien (partie 1/2)
Section : [01:05 - 07:10]
Description : Résumé de la section en quelques phrases.

II. Thème : Un autre sujet, énoncé comme une phrase complète
Section : [07:10 - 15:05]
Description : Résumé de la section.
```

La partie « qui parle » est volontairement laissée de côté.

## Comment ça marche

| Étape | Méthode |
|---|---|
| Transcription | faster-whisper (GPU conseillé), audio 16 kHz passé directement en tableau numpy (évite les erreurs PyAV vues sur Colab) |
| Frontières | Le texte est regroupé en **blocs de 30 s**, transformés en vecteurs. Pour chaque point de coupure possible, on compare **3 minutes avant** et **3 minutes après** : les creux de similarité les plus marqués deviennent des frontières, avec un minimum de 5 min entre deux frontières. |
| Longueur | Durée visée par section (`TARGET_SECTION_SEC`), minimum (`MIN_SECTION_SEC`) et maximum (`MAX_SECTION_SEC`, coupure forcée au-delà). |
| Thème et description | Un **LLM** lit chaque section (en connaissant les thèmes précédents) et rédige thème + description. Il indique aussi si la section **prolonge** la précédente : elles sont alors **fusionnées** (jusqu'au maximum) ou numérotées « partie 1/2, 2/2 ». |
| Sans LLM | Repli : phrases les plus représentatives de la section (résumé extractif). Lisible, mais moins fin. |

## Où tourne quoi

- **Colab (GPU T4)** : Whisper et un LLM open source (Qwen2.5-7B en 4 bits) tournent sur le GPU. Active le GPU : *Exécution → Modifier le type d'exécution → T4*.
- **PC local (sans GPU CUDA)** : Whisper `small`/`medium` sur CPU ; pour le LLM, utilise **Ollama** (`ollama pull qwen2.5:7b-instruct`), détecté automatiquement.
- **API** (option, à activer explicitement dans la configuration) : serveur compatible OpenAI, ou API Anthropic. Le texte de la réunion quitte alors ta machine.

Chaque étape écrit son résultat dans le dossier du projet (`transcript.json`, `sections.json`, `plan_thematique.txt`, `sections_video/`) et **se saute si le résultat existe déjà**.

> **Cadre légal** : enregistrer et republier une réunion demande le consentement des participants. Pour du contenu YouTube, respecte les conditions de la plateforme et le droit d'auteur.

## 1. Configuration
C'est la **seule cellule à modifier**.

In [25]:
# ============================================================
# CONFIGURATION
# ============================================================

# --- Entrée ---------------------------------------------------------------
INPUT = "https://www.youtube.com/watch?v=V_18dl8rJ9s"              # chemin d'un fichier (relatif au dossier du notebook), URL YouTube, ou "" (Colab : téléversement)
TRANSCRIPT_FILE = ""    # optionnel : .vtt / .srt Teams, Zoom... -> remplace Whisper

# --- Étapes ---------------------------------------------------------------
RUN_TRANSCRIBE = True   # 1. transcription (ou import du .vtt)
RUN_THEMES     = True   # 2. sections thématiques + thèmes + descriptions
RUN_VIDEO      = True   # 3. découpage de la vidéo + chapitres
FORCE = set()           # refaire malgré le cache : {"audio", "transcribe", "themes", "video"}
INSTALL_DEPS = True     # installe les paquets manquants

# --- Dossiers -------------------------------------------------------------
WORK_DIR = "outputs"            # en local : transcript.json, sections.json, plan_thematique.txt, sections_video/...
MOUNT_DRIVE = True              # Colab : écrire dans Google Drive (résultats conservés)
DRIVE_SUBDIR = "meeting_pipeline"

# --- Transcription --------------------------------------------------------
LANGUAGE = "fr"          # "fr", "en", ou None (détection automatique)
MULTILINGUAL = False     # True = langue détectée par segment (contenu mêlant FR et EN)
WHISPER_MODEL = "auto"   # auto : large-v3-turbo si GPU, medium sinon (plus fiable que small, ~2-3x plus lent ; "small" pour aller vite)
WORD_TIMESTAMPS = True   # horodatage mot par mot (sous-titres animés, clips calés sur les mots)
INITIAL_PROMPT = "Jésus-Christ, Saint-Esprit, CMCI, CMFI, ZTF, convention, jeûne, réveil, repentance."   # noms propres, lieux, jargon : à compléter

# --- Découpage thématique -------------------------------------------------
MIN_SECTION_SEC = 300      # durée minimale d'une section (5 min)
TARGET_SECTION_SEC = 480   # durée visée (8 min) : augmente-la pour des sections plus longues
MAX_SECTION_SEC = 1200     # au-delà (20 min), coupure forcée ; limite aussi les fusions
BLOCK_SEC = 30             # taille des blocs de texte comparés
TOPIC_WINDOW_SEC = 180     # durée comparée avant/après un point de coupure
TOPIC_STRICTNESS = 0.0     # > 0 : ne garde que les ruptures les plus nettes (sections plus longues)
EMBED_MODEL = "BAAI/bge-m3"   # multilingue ; plus léger : "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# --- Rédaction des thèmes et descriptions (LLM) --------------------------
LLM_BACKEND = "auto"       # "auto" | "transformers" (GPU) | "openai" (Ollama, LM Studio, API compatible) | "anthropic" | "none"
OUTPUT_LANGUAGE = "français"
GLOSSARY = "Jésus-Christ, Saint-Esprit, CMCI, CMFI"   # termes/noms à orthographier correctement dans les textes produits
CHAPTER_TITLE_MAX = 50     # longueur max des titres de chapitres (YouTube, noms des fichiers vidéo)
MERGE_CONTINUING = True    # fusionner les sections consécutives qui traitent du même thème (<= MAX_SECTION_SEC)
THEME_SIM_CONTINUE = 0.88  # similarité (0-1) entre deux thèmes consécutifs au-delà de laquelle c'est une suite, même si le LLM dit non
LLM_MAX_CHARS = 20000      # texte maximal envoyé au LLM par section (début + fin si plus long)

LLM_HF_MODEL = "Qwen/Qwen2.5-7B-Instruct"   # backend "transformers" (4 bits sur GPU)
LLM_4BIT = True
LLM_BASE_URL = "http://localhost:11434/v1"  # backend "openai" : Ollama ; LM Studio : http://localhost:1234/v1
LLM_MODEL = "qwen2.5-7b-12k"            # = qwen2.5:7b-instruct avec contexte 12k (voir README : `ollama create qwen2.5-7b-12k -f Modelfile`). Le contexte par défaut d'Ollama (4k) tronquerait les sections.
LLM_API_KEY = ""                            # vide pour un serveur local (sinon variable/secret OPENAI_API_KEY possible)
ANTHROPIC_MODEL = "claude-sonnet-5-5"       # backend "anthropic" : clé dans ANTHROPIC_API_KEY (variable ou secret Colab)
ANTHROPIC_URL = "https://api.anthropic.com/v1/messages"

# --- Exports ---------------------------------------------------------------
EXPORT_FULL_TEXT = True    # transcription_structuree.md (texte complet par section)
CUT_SECTIONS = True        # un fichier vidéo par section
CUT_MODE = "reencode"      # "reencode" (précis, plus lent) | "copy" (rapide, précis à quelques secondes près)
EMBED_CHAPTERS = True      # copie de la vidéo avec chapitres intégrés

## 2. Environnement et utilitaires

In [26]:
import sys, os, re, json, gc, math, time, shutil, subprocess, unicodedata, importlib, importlib.util, bisect, glob
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

def pip_install(*pkgs):
    print("pip install", *pkgs)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
    importlib.invalidate_caches()

def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except (ImportError, ValueError):
        return False

def detect_torch():
    try:
        import torch
        return torch, bool(torch.cuda.is_available())
    except ImportError:
        return None, False

torch, HAS_CUDA = detect_torch()

# --- paquets nécessaires selon les étapes et le LLM choisis ------------------
need = {"numpy": "numpy", "soundfile": "soundfile", "tqdm": "tqdm", "requests": "requests"}
if RUN_TRANSCRIBE and not TRANSCRIPT_FILE:
    need["faster_whisper"] = "faster-whisper"
if RUN_THEMES:
    need["sentence_transformers"] = "sentence-transformers"
    if LLM_BACKEND == "transformers" or (LLM_BACKEND == "auto" and HAS_CUDA):
        need["transformers"] = "transformers"
        need["accelerate"] = "accelerate"
        if HAS_CUDA and LLM_4BIT:
            need["bitsandbytes"] = "bitsandbytes"
missing = [pkg for mod, pkg in need.items() if not has_module(mod)]
if missing and INSTALL_DEPS:
    pip_install(*missing)
    print("Si Colab/Jupyter signale une incompatibilité de versions, redémarre le noyau puis relance depuis le début.")
elif missing:
    print("Paquets manquants :", missing)
if torch is None:
    torch, HAS_CUDA = detect_torch()

import numpy as np
import requests
from tqdm.auto import tqdm

# --- ffmpeg ---------------------------------------------------------------
if not (shutil.which("ffmpeg") and shutil.which("ffprobe")):
    raise RuntimeError("ffmpeg/ffprobe introuvables. Windows : winget install Gyan.FFmpeg | macOS : brew install ffmpeg | "
                       "Linux : sudo apt install ffmpeg (puis redémarre le noyau).")

DEVICE = "cuda" if HAS_CUDA else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if HAS_CUDA else None
CPU_THREADS = os.cpu_count() or 4
WHISPER_MODEL_NAME = ("large-v3-turbo" if HAS_CUDA else "medium") if WHISPER_MODEL == "auto" else WHISPER_MODEL
CPU_FALLBACK_MODEL = "small"

# --- dossier de travail -----------------------------------------------------
if IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive") / DRIVE_SUBDIR
elif IN_COLAB:
    WORK = Path("/content/work")
else:
    WORK = Path(WORK_DIR)
WORK.mkdir(parents=True, exist_ok=True)
WORK = WORK.resolve()

print(f"Plateforme : {'Colab' if IN_COLAB else 'local'} | calcul : {DEVICE}" + (f" ({GPU_NAME})" if GPU_NAME else ""))
print(f"Dossier de travail : {WORK}")
if not HAS_CUDA:
    print("Pas de GPU CUDA" + (" : active un GPU T4 (Exécution > Modifier le type d'exécution)." if IN_COLAB else " : Whisper tournera sur CPU (plus lent)."))

Plateforme : local | calcul : cpu
Dossier de travail : /home/guillias/Bureau/socialMediaContents/outputs
Pas de GPU CUDA : Whisper tournera sur CPU (plus lent).


In [27]:
# ------------------------- utilitaires généraux ---------------------------
def run(cmd, cwd=None):
    r = subprocess.run([str(c) for c in cmd], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"Commande échouée : {' '.join(map(str, cmd[:4]))} ...\n{r.stderr[-1500:]}")
    return r

def probe(path):
    out = run(["ffprobe", "-v", "error", "-show_entries", "format=duration:stream=codec_type", "-of", "json", path]).stdout
    d = json.loads(out)
    duration = float(d.get("format", {}).get("duration") or 0)
    has_video = any(s.get("codec_type") == "video" for s in d.get("streams", []))
    return duration, has_video

def slugify(s, maxlen=60):
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    s = re.sub(r"[^A-Za-z0-9]+", "-", s).strip("-").lower()
    return (s or "x")[:maxlen].strip("-") or "x"

def fmt_time(t):                       # H:MM:SS (affichage console)
    t = int(round(t))
    return f"{t // 3600}:{t // 60 % 60:02d}:{t % 60:02d}"

def fmt_clock(t, long_fmt):            # [MM:SS] ou [H:MM:SS] pour les vidéos d'une heure ou plus
    t = int(round(t))
    h, m, s = t // 3600, t // 60 % 60, t % 60
    return f"{h}:{m:02d}:{s:02d}" if long_fmt else f"{m:02d}:{s:02d}"

def to_roman(n):
    out = ""
    for v, r in ((1000, "M"), (900, "CM"), (500, "D"), (400, "CD"), (100, "C"), (90, "XC"),
                 (50, "L"), (40, "XL"), (10, "X"), (9, "IX"), (5, "V"), (4, "IV"), (1, "I")):
        while n >= v:
            out += r
            n -= v
    return out

def save_json(path, obj):
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")

def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def is_url(s):
    return bool(re.match(r"^https?://", s.strip()))

def get_secret(name):
    val = os.environ.get(name)
    if not val and IN_COLAB:
        try:
            from google.colab import userdata
            val = userdata.get(name)
        except Exception:
            val = None
    return val

## 3. Source et audio

- Chemin de fichier (par ex. dans ton Drive) : utilisé tel quel. URL YouTube : téléchargée avec `yt-dlp` (YouTube bloque souvent les IP de Colab : télécharge alors en local).
- `INPUT = ""` sur Colab ouvre une fenêtre de téléversement.
- Si `INPUT` pointe vers un `audio.wav` produit par une exécution précédente, son dossier devient le dossier du projet.

In [28]:
def download_url(url):
    try:
        import yt_dlp
    except ImportError:
        pip_install("yt-dlp")
        import yt_dlp
    dl_dir = WORK / "downloads"
    dl_dir.mkdir(exist_ok=True)
    opts = {"outtmpl": str(dl_dir / "%(id)s.%(ext)s"), "format": "bv*[height<=1080]+ba/b[height<=1080]/b",
            "merge_output_format": "mp4", "noplaylist": True, "quiet": True, "no_warnings": True}
    with yt_dlp.YoutubeDL(opts) as ydl:
        info = ydl.extract_info(url, download=True)
        try:
            path = Path(info["requested_downloads"][0]["filepath"])
        except (KeyError, IndexError):
            path = Path(ydl.prepare_filename(info)).with_suffix(".mp4")
    return path.resolve()

def resolve_source():
    src = INPUT.strip().strip('"').strip("'")
    if not src:
        if IN_COLAB:
            from google.colab import files
            up = files.upload()
            if not up:
                raise RuntimeError("Aucun fichier téléversé.")
            name = next(iter(up))
            dest = WORK / "uploads"
            dest.mkdir(exist_ok=True)
            shutil.move(name, dest / name)
            return (dest / name).resolve()
        raise ValueError("Renseigne INPUT (chemin de fichier ou URL YouTube) dans la cellule de configuration.")
    if is_url(src):
        return download_url(src)
    p = Path(src).expanduser()
    if not p.exists():
        raise FileNotFoundError(p)
    return p.resolve()

SOURCE = resolve_source()
PROJECT = SOURCE.parent if SOURCE.name == "audio.wav" else WORK / slugify(SOURCE.stem)
PROJECT.mkdir(parents=True, exist_ok=True)
DURATION, HAS_VIDEO = probe(SOURCE)
AUDIO = PROJECT / "audio.wav"
TRANSCRIPT_JSON = PROJECT / "transcript.json"
SECTIONS_JSON = PROJECT / "sections.json"

if RUN_TRANSCRIBE and not TRANSCRIPT_FILE and not TRANSCRIPT_JSON.exists() or "audio" in FORCE:
    if SOURCE != AUDIO and (not AUDIO.exists() or "audio" in FORCE):
        print("Extraction audio (mono, 16 kHz)...")
        run(["ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-i", SOURCE, "-vn", "-ac", "1", "-ar", "16000",
             "-c:a", "pcm_s16le", AUDIO])

print(f"Source : {SOURCE.name}")
print(f"Projet : {PROJECT}")
print(f"Durée  : {fmt_time(DURATION)} | vidéo : {'oui' if HAS_VIDEO else 'non (audio seul : découpage vidéo impossible)'}")

Extraction audio (mono, 16 kHz)...                          
Source : V_18dl8rJ9s.mp4
Projet : /home/guillias/Bureau/socialMediaContents/outputs/v-18dl8rj9s
Durée  : 1:51:12 | vidéo : oui


## 4. Transcription

Priorité : `transcript.json` existant (cache) → import `TRANSCRIPT_FILE` (`.vtt` Teams/Zoom ou `.srt`) → faster-whisper.

Sur GPU, si les bibliothèques CUDA sont introuvables (erreur `libcublas` / `libcudnn` fréquente sur Colab), le notebook les précharge,
installe au besoin les paquets `nvidia-*-cu12`, puis retombe sur le CPU en dernier recours.

In [29]:
# ------------------------- import .vtt / .srt ------------------------------
def parse_vtt(path):
    text = Path(path).read_text(encoding="utf-8-sig", errors="replace").replace("\r\n", "\n")
    ts = re.compile(r"(?:(\d+):)?(\d{1,2}):(\d{2})[.,](\d{3})\s*-->\s*(?:(\d+):)?(\d{1,2}):(\d{2})[.,](\d{3})")
    sec = lambda h, m, s, ms: int(h or 0) * 3600 + int(m) * 60 + int(s) + int(ms) / 1000
    units = []
    for block in re.split(r"\n\s*\n", text):
        lines = block.strip().split("\n")
        idx = next((i for i, l in enumerate(lines) if "-->" in l), None)
        m = ts.search(lines[idx]) if idx is not None else None
        if not m:
            continue
        body = " ".join(l.strip() for l in lines[idx + 1:] if l.strip())
        stripped = re.sub(r"^<v\s+[^>]+>", "", body)                # Teams : <v Nom>texte</v>
        if stripped == body:
            stripped = re.sub(r"^[^:<>]{1,60}:\s+", "", body)       # Zoom : Nom: texte
        body = re.sub(r"<[^>]+>", "", stripped).strip()
        if body:
            units.append({"start": sec(*m.group(1, 2, 3, 4)), "end": sec(*m.group(5, 6, 7, 8)), "text": body})
    return units

def merge_cues(units, max_gap=1.5, max_len=20.0):
    out = []
    for u in units:
        p = out[-1] if out else None
        if p and u["start"] - p["end"] <= max_gap and u["end"] - p["start"] <= max_len:
            p["end"] = u["end"]
            p["text"] += " " + u["text"]
        else:
            out.append(dict(u))
    return out

# ------------------------- Whisper ---------------------------------------
def preload_cuda_libs():
    if not sys.platform.startswith("linux"):
        return
    import ctypes, site
    roots = set()
    try:
        roots |= {os.path.join(sp, "nvidia") for sp in site.getsitepackages() + [site.getusersitepackages()]}
    except Exception:
        pass
    for pattern in ("libcublasLt.so*", "libcublas.so*", "libcudnn.so*"):
        for root in roots:
            for path in sorted(glob.glob(os.path.join(root, "*", "lib", pattern))):
                try:
                    ctypes.CDLL(path, mode=ctypes.RTLD_GLOBAL)
                except OSError:
                    pass

def _whisper_pass(audio, duration, model_name, device, compute):
    from faster_whisper import WhisperModel
    print(f"Whisper '{model_name}' ({device}, {compute})...")
    model = WhisperModel(model_name, device=device, compute_type=compute, cpu_threads=CPU_THREADS)
    kwargs = dict(language=LANGUAGE, vad_filter=True, vad_parameters=dict(min_silence_duration_ms=500),
                  beam_size=5, condition_on_previous_text=False, initial_prompt=INITIAL_PROMPT or None,
                  word_timestamps=WORD_TIMESTAMPS)
    if MULTILINGUAL:
        kwargs["multilingual"] = True
    segs, done = [], 0
    try:
        try:
            seg_iter, info = model.transcribe(audio, **kwargs)
        except TypeError:                          # faster-whisper sans l'option multilingual
            kwargs.pop("multilingual", None)
            seg_iter, info = model.transcribe(audio, **kwargs)
        with tqdm(total=max(1, int(duration)), unit="s", desc="Transcription") as bar:
            for s in seg_iter:
                seg = {"start": round(float(s.start), 2), "end": round(float(s.end), 2), "text": s.text.strip()}
                if WORD_TIMESTAMPS and s.words:
                    seg["words"] = [{"w": w.word.strip(), "s": round(float(w.start), 2), "e": round(float(w.end), 2)}
                                    for w in s.words if w.word.strip()]
                segs.append(seg)
                step = int(min(s.end, duration)) - done
                if step > 0:
                    bar.update(step)
                    done += step
        lang = info.language
    finally:
        del model
        gc.collect()
        if HAS_CUDA:
            torch.cuda.empty_cache()
    return segs, lang, model_name

def run_whisper(audio_path, duration):
    import soundfile as sf
    wav, sr = sf.read(str(audio_path), dtype="float32")      # tableau numpy : pas de décodage PyAV
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    device, compute, name = DEVICE, ("float16" if DEVICE == "cuda" else "int8"), WHISPER_MODEL_NAME
    if device == "cuda":
        preload_cuda_libs()
    for attempt in range(3):
        try:
            return _whisper_pass(wav, duration, name, device, compute)
        except Exception as e:
            msg = str(e).lower()
            if not (device == "cuda" and any(k in msg for k in ("cublas", "cudnn", "cuda", "libcu"))):
                raise
            print("Problème GPU/CUDA :", str(e)[:200])
            if attempt == 0 and INSTALL_DEPS:
                pip_install("nvidia-cublas-cu12", "nvidia-cudnn-cu12")
                preload_cuda_libs()
                continue
            print(f"Repli sur le CPU avec le modèle '{CPU_FALLBACK_MODEL}'.")
            device, compute, name = "cpu", "int8", CPU_FALLBACK_MODEL
    raise RuntimeError("Transcription impossible.")

HALLU = re.compile(r"(sous-titr|amara\.org|merci d.avoir regard|abonnez-vous|thanks for watching|subtitles by)", re.I)

def clean_segments(segs):
    out = []
    for s in segs:
        t = s["text"].strip()
        if not t or (HALLU.search(t) and len(t) < 80):
            continue
        if len(out) >= 2 and t == out[-1]["text"] == out[-2]["text"]:      # boucle de répétition
            continue
        item = {"start": s["start"], "end": s["end"], "text": t}
        if s.get("words"):
            item["words"] = s["words"]
        out.append(item)
    return out

In [30]:
if RUN_TRANSCRIBE:
    if TRANSCRIPT_JSON.exists() and "transcribe" not in FORCE:
        print("Transcription déjà présente : réutilisée (FORCE = {'transcribe'} pour la refaire).")
    elif TRANSCRIPT_FILE:
        units = merge_cues(parse_vtt(TRANSCRIPT_FILE))
        save_json(TRANSCRIPT_JSON, {"meta": {"origin": "vtt", "duration": DURATION, "language": LANGUAGE}, "units": units})
    else:
        segs, lang, used_model = run_whisper(AUDIO, DURATION)
        units = clean_segments(segs)
        save_json(TRANSCRIPT_JSON, {"meta": {"origin": "whisper", "whisper_model": used_model,
                                             "duration": DURATION, "language": lang,
                                             "word_timestamps": any("words" in u for u in units)}, "units": units})

if not TRANSCRIPT_JSON.exists():
    raise FileNotFoundError(f"{TRANSCRIPT_JSON} introuvable : lance l'étape 1 (RUN_TRANSCRIBE = True) ou copie le fichier ici.")
TRANSCRIPT = load_json(TRANSCRIPT_JSON)
UNITS = TRANSCRIPT["units"]
if WORD_TIMESTAMPS and TRANSCRIPT["meta"].get("origin") == "whisper" and not any("words" in u for u in UNITS):
    print("Transcription en cache sans horodatage par mot (modèle %s) : FORCE = {'transcribe'} pour la refaire "
          "avec le modèle et le glossaire actuels." % TRANSCRIPT["meta"].get("whisper_model"))
nwords = sum(len(u["text"].split()) for u in UNITS)
print(f"{len(UNITS)} segments, {nwords} mots | origine : {TRANSCRIPT['meta'].get('origin')} | langue : {TRANSCRIPT['meta'].get('language')}")
for u in UNITS[:5]:
    print(f"[{fmt_time(u['start'])}] {u['text'][:120]}")

Whisper 'medium' (cpu, int8)...


Transcription:   0%|          | 0/6671 [00:00<?, ?s/s]

919 segments, 6648 mots | origine : whisper | langue : fr
[0:00:00] Amen. C'est votre tour maintenant. Prenons dix personnes par rangée, dix comme dix, et soyez brefs s'il vous plait.
[0:00:19] Ok, tous ceux qui sont déjà debout, pour aujourd'hui, soyons indulgents.
[0:00:29] Mais priez une seule phrase seulement.
[0:00:33] Alleluia! Amen! Alleluia! Amen!
[0:00:38] euh, l'us


## 5. Frontières des sections

1. Le texte est regroupé en **blocs de `BLOCK_SEC` secondes**, puis vectorisé.
2. Pour chaque point de coupure possible, on calcule la ressemblance entre `TOPIC_WINDOW_SEC` avant et après ; un **creux** de ressemblance indique un changement de sujet.
3. On retient les creux les plus nets, **au plus un toutes les `MIN_SECTION_SEC`**, jusqu'à obtenir une section tous les `TARGET_SECTION_SEC` environ. Une section dépassant `MAX_SECTION_SEC` est coupée de force à son meilleur creux.
4. Chaque frontière est ensuite **calée sur la pause ou la fin de phrase la plus proche** (± `BLOCK_SEC`).

Le LLM pourra ensuite **fusionner** deux sections consécutives qui traitent du même thème (étape 6).

In [31]:
def make_blocks(units, block_sec):
    blocks, cur, t0 = [], None, units[0]["start"]
    for i, u in enumerate(units):
        k = int((u["start"] - t0) // block_sec)
        if cur is None or k != cur["k"]:
            cur = {"k": k, "a": i, "b": i + 1, "start": u["start"], "text": u["text"]}
            blocks.append(cur)
        else:
            cur["b"] = i + 1
            cur["text"] += " " + u["text"]
    return blocks

def depth_scores(emb, weights, window):
    n = len(emb)
    sims = np.ones(n)
    for i in range(1, n):
        a, b = max(0, i - window), min(n, i + window)
        L = (emb[a:i] * weights[a:i, None]).sum(0)
        R = (emb[i:b] * weights[i:b, None]).sum(0)
        sims[i] = float(L @ R / (np.linalg.norm(L) * np.linalg.norm(R) + 1e-9))
    if n > 1:
        sims[0] = sims[1]
    sm = np.convolve(np.pad(sims, (1, 1), mode="edge"), np.ones(3) / 3, mode="valid")
    depth = np.zeros(n)
    for i in range(1, n):
        l = i
        while l > 0 and sm[l - 1] >= sm[l]:
            l -= 1
        r = i
        while r < n - 1 and sm[r + 1] >= sm[r]:
            r += 1
        depth[i] = (sm[l] - sm[i]) + (sm[r] - sm[i])
    return depth

def pick_boundaries(depth, btimes, t_begin, t_end, min_gap, target, max_len, strictness):
    n = len(depth)
    def admissible(t, chosen):
        return t - t_begin >= min_gap and t_end - t >= min_gap and all(abs(t - btimes[j]) >= min_gap for j in chosen)
    if n < 3:
        return []
    peaks = [i for i in range(1, n) if depth[i] > 0 and depth[i] >= depth[i - 1] and (i == n - 1 or depth[i] >= depth[i + 1])]
    floor = depth[1:].mean() + strictness * depth[1:].std()
    n_target = max(0, int(round((t_end - t_begin) / target)) - 1)
    chosen = []
    for i in sorted(peaks, key=lambda i: -depth[i]):
        if len(chosen) >= n_target or depth[i] < floor:
            break
        if admissible(btimes[i], chosen):
            chosen.append(i)
    while True:                                        # coupures forcées des sections trop longues
        times = sorted([t_begin] + [btimes[j] for j in chosen] + [t_end])
        too_long = [(a, b) for a, b in zip(times[:-1], times[1:]) if b - a > max_len]
        if not too_long:
            break
        a, b = max(too_long, key=lambda ab: ab[1] - ab[0])
        inside = [i for i in range(1, n) if a + min_gap <= btimes[i] <= b - min_gap]
        if inside:
            i = max(inside, key=lambda i: depth[i])
        else:
            mid = (a + b) / 2
            i = min(range(1, n), key=lambda i: abs(btimes[i] - mid))
        if i in chosen:
            break
        chosen.append(i)
    return sorted(chosen)

def snap_to_pause(units, starts, t, radius):
    lo = bisect.bisect_left(starts, t - radius)
    best, best_score = None, -1e9
    for j in range(max(1, lo), len(units)):
        dt = starts[j] - t
        if dt > radius:
            break
        gap = max(0.0, units[j]["start"] - units[j - 1]["end"])
        bonus = 0.5 if units[j - 1]["text"].rstrip().endswith((".", "?", "!", "…")) else 0.0
        score = gap + bonus - 0.01 * abs(dt)
        if score > best_score:
            best, best_score = j, score
    return best

def section_dict(units, a, b):
    return {"a": a, "b": b, "start": units[a]["start"], "end": units[b - 1]["end"]}

EMBEDDER = None
def get_embedder():
    global EMBEDDER
    if EMBEDDER is None:
        from sentence_transformers import SentenceTransformer
        EMBEDDER = SentenceTransformer(EMBED_MODEL, device=DEVICE)
        try:
            EMBEDDER.max_seq_length = min(512, EMBEDDER.max_seq_length or 512)
        except Exception:
            pass
    return EMBEDDER

def embed_texts(texts, show=False):
    return np.asarray(get_embedder().encode(texts, batch_size=32, normalize_embeddings=True, show_progress_bar=show))

def initial_sections(units):
    t_begin, t_end = units[0]["start"], units[-1]["end"]
    blocks = make_blocks(units, BLOCK_SEC)
    if len(blocks) < 4 or t_end - t_begin < 2 * MIN_SECTION_SEC:
        return [section_dict(units, 0, len(units))]
    print(f"{len(blocks)} blocs de {BLOCK_SEC} s : calcul des vecteurs...")
    emb = embed_texts([b["text"] for b in blocks], show=True)
    weights = np.array([max(1, len(b["text"].split())) for b in blocks], dtype=float)
    btimes = np.array([b["start"] for b in blocks])
    depth = depth_scores(emb, weights, max(2, int(round(TOPIC_WINDOW_SEC / BLOCK_SEC))))
    picked = pick_boundaries(depth, btimes, t_begin, t_end, MIN_SECTION_SEC, TARGET_SECTION_SEC, MAX_SECTION_SEC, TOPIC_STRICTNESS)
    starts = [u["start"] for u in units]
    cuts = []
    for i in picked:
        j = snap_to_pause(units, starts, blocks[i]["start"], BLOCK_SEC)
        cuts.append(j if j is not None else blocks[i]["a"])
    cuts = sorted(set(c for c in cuts if 0 < c < len(units)))
    bounds = [0] + cuts + [len(units)]
    return [section_dict(units, a, b) for a, b in zip(bounds[:-1], bounds[1:]) if b > a]

## 6. Thèmes et descriptions rédigés par un LLM

Choix du moteur (`LLM_BACKEND`) :

- `auto` : GPU détecté (Colab) → modèle open source `transformers` ; sinon Ollama s'il répond sur `LLM_BASE_URL` ; sinon pas de LLM.
- `transformers` : Qwen2.5-7B-Instruct en 4 bits sur le GPU (premier téléchargement d'environ 15 Go).
- `openai` : tout serveur compatible OpenAI (Ollama, LM Studio, llama.cpp, services cloud). `anthropic` : API Anthropic.
  **Ces deux options cloud envoient le texte de la réunion à un tiers** : à choisir en connaissance de cause.

Pour chaque section, le LLM reçoit le texte et les **thèmes précédents**, et renvoie un thème (une phrase), un **titre court** de chapitre
(≤ `CHAPTER_TITLE_MAX` caractères), une description et indique si elle **prolonge directement la section précédente**
(fusion, ou « partie 1/2, 2/2 » si la fusion dépasse `MAX_SECTION_SEC`). Chaque section garde son propre thème.
Sans LLM, un résumé extractif prend le relais.

In [32]:
LLM = {"mode": "none", "tok": None, "model": None}

def openai_headers():
    key = LLM_API_KEY or get_secret("OPENAI_API_KEY") or ""
    return {"Content-Type": "application/json", **({"Authorization": f"Bearer {key}"} if key else {})}

def openai_up():
    try:
        return requests.get(LLM_BASE_URL.rstrip("/") + "/models", headers=openai_headers(), timeout=3).ok
    except Exception:
        return False

def load_hf_llm():
    import transformers
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from packaging.version import Version
    dtype_key = "dtype" if Version(transformers.__version__.split("+")[0]) >= Version("4.56") else "torch_dtype"
    print(f"Chargement de {LLM_HF_MODEL} ({'4 bits' if HAS_CUDA and LLM_4BIT else 'précision réduite'}) ...")
    tok = AutoTokenizer.from_pretrained(LLM_HF_MODEL)
    kwargs = {"device_map": "auto"}
    if HAS_CUDA and LLM_4BIT:
        from transformers import BitsAndBytesConfig
        kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                           bnb_4bit_compute_dtype=torch.float16)
    else:
        kwargs[dtype_key] = torch.float16 if HAS_CUDA else torch.bfloat16
    model = AutoModelForCausalLM.from_pretrained(LLM_HF_MODEL, **kwargs)
    model.eval()
    return tok, model

def init_llm():
    mode = LLM_BACKEND
    if mode == "auto":
        mode = "transformers" if HAS_CUDA else ("openai" if openai_up() else "none")
    if mode == "openai" and not openai_up():
        print(f"Serveur LLM injoignable ({LLM_BASE_URL}) : repli sur le résumé extractif.")
        mode = "none"
    if mode == "anthropic" and not get_secret("ANTHROPIC_API_KEY"):
        print("Clé ANTHROPIC_API_KEY introuvable : repli sur le résumé extractif.")
        mode = "none"
    if mode == "transformers":
        try:
            LLM["tok"], LLM["model"] = load_hf_llm()
        except Exception as e:
            print("Chargement du LLM impossible :", str(e)[:300], "\nRepli sur le résumé extractif.")
            mode = "none"
    LLM["mode"] = mode
    print("Moteur de rédaction :", {"none": "aucun (résumé extractif)"}.get(mode, mode))

def free_llm():
    LLM["tok"] = LLM["model"] = None
    gc.collect()
    if HAS_CUDA:
        torch.cuda.empty_cache()

def llm_call(prompt, max_tokens=700):
    mode = LLM["mode"]
    if mode == "transformers":
        tok, model = LLM["tok"], LLM["model"]
        enc = tok.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True,
                                      return_tensors="pt", return_dict=True)
        enc = {k: v.to(model.device) for k, v in enc.items()}
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_tokens, do_sample=False)
        return tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
    if mode == "openai":
        r = requests.post(LLM_BASE_URL.rstrip("/") + "/chat/completions", headers=openai_headers(), timeout=1200,
                          json={"model": LLM_MODEL, "temperature": 0.2, "max_tokens": max_tokens,
                                "messages": [{"role": "user", "content": prompt}]})
        r.raise_for_status()
        return r.json()["choices"][0]["message"]["content"]
    if mode == "anthropic":
        r = requests.post(ANTHROPIC_URL, timeout=600,
                          headers={"x-api-key": get_secret("ANTHROPIC_API_KEY"), "anthropic-version": "2023-06-01",
                                   "content-type": "application/json"},
                          json={"model": ANTHROPIC_MODEL, "max_tokens": max_tokens, "temperature": 0.2,
                                "messages": [{"role": "user", "content": prompt}]})
        r.raise_for_status()
        return "".join(b.get("text", "") for b in r.json()["content"] if b.get("type") == "text")
    raise RuntimeError("Aucun LLM actif")

# ------------------------- prompt et lecture de la réponse -----------------
def build_prompt(text, ctx_titles, k, n, minutes):
    start = max(0, len(ctx_titles) - 8)
    prev = "\n".join(f"{j + 1}. {ctx_titles[j]}" for j in range(start, len(ctx_titles))) or "(aucune : c'est le début de l'enregistrement)"
    gloss = f"\nTermes et noms à orthographier ainsi : {GLOSSARY}\n" if GLOSSARY.strip() else ""
    p = len(ctx_titles)                      # numéro de la section immédiatement précédente
    suite = (f'- "suite_de" : {p} UNIQUEMENT si cette section poursuit sans rupture le sujet précis de la section {p} '
             f"(la même idée se prolonge) ; sinon null. Un sujet voisin, un même thème général (foi, réveil, Saint-Esprit...) "
             f"ou le retour à une idée évoquée plus tôt ne sont PAS une suite. Dans le doute : null."
             if p else '- "suite_de" : null (première section).')
    return f"""Tu structures la transcription d'un enregistrement (enseignement, prédication ou réunion) en grandes sections thématiques.
Chaque section dure plusieurs minutes : le thème doit exprimer l'idée centrale de TOUTE la section, pas un détail.

Section à analyser : n°{k} sur environ {n} (durée : {minutes} min).{gloss}
Thèmes des sections précédentes :
{prev}

TRANSCRIPTION (reconnaissance vocale automatique, avec d'éventuelles erreurs) :
<<<
{text}
>>>

Rends un objet JSON avec exactement ces clés :
- "theme" : UNE phrase complète et compréhensible (10 à 22 mots), en {OUTPUT_LANGUAGE}, qui énonce l'idée centrale de la section. Pas de liste de mots-clés, pas de titre télégraphique, pas de numérotation.
- "titre_court" : titre de chapitre YouTube en {OUTPUT_LANGUAGE}, {CHAPTER_TITLE_MAX} caractères maximum, concret et propre à CETTE section (ce qui la distingue des autres sections), sans numérotation, sans guillemets, sans point final.
- "description" : résumé en {OUTPUT_LANGUAGE} de 3 à 5 phrases : idées principales, arguments, exemples et références (textes, personnes, chiffres) effectivement cités. N'invente rien qui ne soit pas dans le texte.
{suite}

Réponds UNIQUEMENT avec l'objet JSON."""

def parse_llm_json(txt):
    m = re.search(r"\{.*\}", txt, re.S)
    if m:
        try:
            d = json.loads(m.group(0))
            if isinstance(d, dict):
                return d
        except Exception:
            pass
    th = re.search(r'"theme"\s*:\s*"((?:[^"\\]|\\.)*)"', txt, re.S)
    ds = re.search(r'"description"\s*:\s*"((?:[^"\\]|\\.)*)', txt, re.S)
    if th and ds:
        return {"theme": th.group(1), "description": ds.group(1).rstrip('"} \n'), "suite_de": None}
    return None

def clean_text(s):
    s = re.sub(r"\s+", " ", str(s or "")).strip().strip('"«» ')
    return (s[:1].upper() + s[1:]) if s else s

def section_text(units, sec):
    text = " ".join(u["text"] for u in units[sec["a"]:sec["b"]])
    if len(text) > LLM_MAX_CHARS:
        h = int(LLM_MAX_CHARS * 0.5)
        text = text[:h] + "\n[...]\n" + text[-(LLM_MAX_CHARS - h):]
    return text

# ------------------------- repli extractif (sans LLM) ------------------------
def extractive_describe(units, sec):
    full = " ".join(u["text"] for u in units[sec["a"]:sec["b"]])
    sents = [s.strip() for s in re.split(r"(?<=[.!?…])\s+", full) if 6 <= len(s.split()) <= 45]
    if not sents:
        sents = [u["text"] for u in units[sec["a"]:sec["b"]] if u["text"].strip()] or [full[:200]]
    if len(sents) > 300:
        sents = sents[:: math.ceil(len(sents) / 300)]
    E = embed_texts(sents)
    score = E @ E.mean(0)
    theme = sents[int(np.argmax(score))]
    words = theme.split()
    if len(words) > 24:
        theme = " ".join(words[:24]) + "…"
    top = sorted(np.argsort(-score)[:4])
    return {"theme": clean_text(theme), "short_title": "", "description": " ".join(sents[i] for i in top),
            "suite_de": None, "method": "extractif"}

def describe_section(units, sec, ctx_titles, k, n):
    if LLM["mode"] != "none":
        prompt = build_prompt(section_text(units, sec), ctx_titles, k, n, round((sec["end"] - sec["start"]) / 60))
        for attempt in range(2):
            try:
                d = parse_llm_json(llm_call(prompt if attempt == 0 else prompt + "\nAttention : réponds uniquement par le JSON demandé."))
            except Exception as e:
                print("  (LLM) échec :", str(e)[:150])
                d = None
            if d and clean_text(d.get("theme")) and clean_text(d.get("description")):
                sd = d.get("suite_de")
                sd = int(sd) if isinstance(sd, (int, float)) or str(sd).strip().isdigit() else None
                return {"theme": clean_text(d["theme"]), "short_title": clean_text(d.get("titre_court")).rstrip(". "),
                        "description": clean_text(d["description"]), "suite_de": sd, "method": "llm"}
        print("  (LLM) réponse inexploitable : repli extractif pour cette section.")
    return extractive_describe(units, sec)

In [33]:
def assemble_sections(units, raw):
    results = []
    for idx, s in enumerate(raw, 1):
        themes = [r["theme"] for r in results]
        d = describe_section(units, s, themes, idx, len(raw))
        prev = results[-1] if results else None
        continues = prev is not None and d["suite_de"] == len(results)      # suite directe de la section précédente uniquement
        if prev is not None and not continues:                               # garde-fou : thèmes quasi identiques
            sim = float(np.dot(*embed_texts([prev["theme"], d["theme"]])))
            if sim >= THEME_SIM_CONTINUE:
                continues = True
                print(f"  section {idx} : thème quasi identique au précédent (similarité {sim:.2f}) -> suite")
        if MERGE_CONTINUING and continues and s["end"] - prev["start"] <= MAX_SECTION_SEC:
            merged = {**prev, "end": s["end"], "b": s["b"]}                # même thème : on fusionne puis on rédige à nouveau
            d2 = describe_section(units, merged, themes[:-1], len(results), len(raw))
            n_prev = len(results) - 1                                      # sections avant la section fusionnée
            merged.update(theme=d2["theme"], short_title=d2["short_title"], description=d2["description"], method=d2["method"],
                          part_of=(n_prev - 1) if n_prev and d2["suite_de"] == n_prev else prev.get("part_of"))
            results[-1] = merged
            print(f"  section {idx}/{len(raw)} fusionnée avec la précédente")
        else:
            results.append({**s, "theme": d["theme"], "short_title": d["short_title"], "description": d["description"],
                            "method": d["method"], "part_of": len(results) - 1 if continues else None})
            print(f"  section {idx}/{len(raw)} : {fmt_time(s['start'])}-{fmt_time(s['end'])}")
    return finalize_titles(results)

def short_title_of(r):
    t = clean_text(r.get("short_title")) or r["theme"]
    if len(t) > CHAPTER_TITLE_MAX:                                         # coupe au dernier mot entier
        t = t[:CHAPTER_TITLE_MAX - 1].rsplit(" ", 1)[0].rstrip(",;:-–— ") + "…"
    return t

def finalize_titles(sections):
    """Titres affichés. Chaque section garde SON thème ; « partie k/m » seulement pour une suite de sections
    consécutives sur le même sujet (part_of = section immédiatement précédente)."""
    chains = []
    for i, r in enumerate(sections):
        if chains and r.get("part_of") == i - 1:
            chains[-1].append(i)
        else:
            chains.append([i])
    for chain in chains:
        m = len(chain)
        for rank, i in enumerate(chain, 1):
            r = sections[i]
            r["id"] = i + 1
            r["theme_display"] = r["theme"] + (f" (partie {rank}/{m})" if m > 1 else "")
            r["chapter_title"] = short_title_of(r) + (f" ({rank}/{m})" if m > 1 else "")
    return sections

def build_plan(sections, long_fmt):
    lines = []
    for r in sections:
        lines += [f"{to_roman(r['id'])}. Thème : {r['theme_display']}",
                  f"Section : [{fmt_clock(r['start'], long_fmt)} - {fmt_clock(r['end'], long_fmt)}]",
                  f"Description : {r['description']}", ""]
    return "\n".join(lines)

PLAN_TXT = PROJECT / "plan_thematique.txt"
SECTIONS = []
LONG_FMT = DURATION >= 3600

if RUN_THEMES and not (SECTIONS_JSON.exists() and "themes" not in FORCE):
    raw = initial_sections(UNITS)
    print(f"{len(raw)} sections initiales (durées : {[round((s['end'] - s['start']) / 60, 1) for s in raw]} min)")
    init_llm()
    SECTIONS = assemble_sections(UNITS, raw)
    free_llm()
    save_json(SECTIONS_JSON, {"source": SOURCE.name, "duration": DURATION,
                              "params": {"min": MIN_SECTION_SEC, "target": TARGET_SECTION_SEC, "max": MAX_SECTION_SEC,
                                         "embed_model": EMBED_MODEL, "llm": LLM["mode"]},
                              "sections": [{k: v for k, v in r.items()} for r in SECTIONS]})
    PLAN_TXT.write_text(build_plan(SECTIONS, LONG_FMT), encoding="utf-8")
elif SECTIONS_JSON.exists():
    SECTIONS = finalize_titles(load_json(SECTIONS_JSON)["sections"])
    PLAN_TXT.write_text(build_plan(SECTIONS, LONG_FMT), encoding="utf-8")
    print("Sections déjà présentes : réutilisées (FORCE = {'themes'} pour les refaire).")

if SECTIONS:
    print("\n" + build_plan(SECTIONS, LONG_FMT))
    print(f"Plan enregistré : {PLAN_TXT}")

198 blocs de 30 s : calcul des vecteurs...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

12 sections initiales (durées : [9.2, 9.3, 9.6, 8.6, 5.2, 13.1, 9.5, 6.1, 6.4, 5.1, 10.8, 15.2] min)
Moteur de rédaction : openai
  section 1/12 : 0:00:00-0:09:09
  section 2/12 : 0:09:20-0:18:39
  section 3 : thème quasi identique au précédent (similarité 0.95) -> suite
  section 3/12 fusionnée avec la précédente
  section 4/12 : 0:29:17-0:37:51
  section 5/12 : 0:37:53-0:43:02
  section 6/12 : 0:43:51-0:56:56
  section 7/12 : 0:57:02-1:06:31
  section 8/12 : 1:06:39-1:12:42
  section 9/12 : 1:12:55-1:19:21
  section 10/12 : 1:19:24-1:24:29
  section 11/12 : 1:24:29-1:35:16
  section 12/12 : 1:35:28-1:50:41

I. Thème : Paul explique les différents types d'appel de Dieu dans la lettre aux Romains
Section : [0:00:00 - 0:09:09]
Description : Paul décrit quatre occurrences de l'appel de Dieu dans sa lettre aux Romains. Il parle des appels généraux à toutes les nations, et spécifiquement à ceux qui ont cru en Jésus-Christ pour la foi et l'obéissance.

II. Thème : L'appartenance à Jésus et 

## 7. Texte complet par section (optionnel)

Écrit `transcription_structuree.md` : le plan, puis le texte de chaque section avec des repères de temps.

In [34]:
if EXPORT_FULL_TEXT and SECTIONS:
    lines = [f"# Transcription structurée — {SOURCE.name}", "", f"Durée : {fmt_time(DURATION)}", "", "## Plan", ""]
    for r in SECTIONS:
        lines.append(f"- **{to_roman(r['id'])}.** {r['theme_display']} `[{fmt_clock(r['start'], LONG_FMT)} - {fmt_clock(r['end'], LONG_FMT)}]`")
    for r in SECTIONS:
        lines += ["", f"## {to_roman(r['id'])}. {r['theme_display']}", "",
                  f"> **Section** : {fmt_clock(r['start'], LONG_FMT)} - {fmt_clock(r['end'], LONG_FMT)}  ", f"> **Description** : {r['description']}", ""]
        para = []
        for u in UNITS[r["a"]:r["b"]]:
            para.append(u["text"])
            if len(para) >= 6:
                lines.append(f"`[{fmt_clock(u['start'], LONG_FMT)}]` " + " ".join(para) + "\n")
                para = []
        if para:
            lines.append(f"`[{fmt_clock(UNITS[r['b'] - 1]['start'], LONG_FMT)}]` " + " ".join(para) + "\n")
    (PROJECT / "transcription_structuree.md").write_text("\n".join(lines), encoding="utf-8")
    print("Exporté :", PROJECT / "transcription_structuree.md")

Exporté : /home/guillias/Bureau/socialMediaContents/outputs/v-18dl8rj9s/transcription_structuree.md


## 8. Découpage de la vidéo et chapitres

À partir de `sections.json`, les sections deviennent **contiguës** (la première commence à 0, chaque section s'arrête là où la suivante commence), donc la vidéo est entièrement couverte, sans trou ni chevauchement.

Sont produits :
- `sections_video/NN_<thème>.mp4` : un fichier par section (`CUT_MODE = "reencode"` : coupe exacte ; `"copy"` : instantané mais calé sur les images clés) ;
- `chapitres_youtube.txt` (titres courts) : à coller dans la description d'une vidéo YouTube (chapitres automatiques) ;
- `chapitres.ffmetadata` et `<nom>_chapitres.mp4` : la vidéo d'origine avec chapitres intégrés (lecteurs VLC, etc.), sans ré-encodage.

Pour un **flux en direct**, enregistre-le d'abord (OBS, `yt-dlp --live-from-start`...) puis fournis le fichier obtenu.

In [35]:
def video_bounds(sections, duration):
    out = []
    for i, s in enumerate(sections):
        a = 0.0 if i == 0 else float(s["start"])
        b = float(duration) if i == len(sections) - 1 else float(sections[i + 1]["start"])
        out.append((a, b))
    return out

def yt_ts(t):
    t = int(round(t))
    return f"{t // 3600}:{t // 60 % 60:02d}:{t % 60:02d}" if t >= 3600 else f"{t // 60}:{t % 60:02d}"

def ffmeta_escape(s):
    return re.sub(r"([=;#\\\n])", lambda m: "\\" + m.group(1), s.replace("\n", " "))

VIDEO_DIR = PROJECT / "sections_video"

if RUN_VIDEO and not HAS_VIDEO:
    print("Pas de piste vidéo : découpage ignoré.")
elif RUN_VIDEO and not SECTIONS:
    print("Aucune section : lance l'étape 2 (RUN_THEMES = True).")
elif RUN_VIDEO:
    bounds = video_bounds(SECTIONS, DURATION)
    keep_ext = SOURCE.suffix.lower() in (".mp4", ".mkv", ".mov", ".m4v", ".webm")
    out_ext = ".mp4" if CUT_MODE == "reencode" else (SOURCE.suffix.lower() if keep_ext else ".mkv")

    # --- chapitres (texte YouTube + métadonnées ffmpeg) ---------------------------------
    (PROJECT / "chapitres_youtube.txt").write_text(
        "\n".join(f"{yt_ts(a)} {r['chapter_title']}" for (a, _), r in zip(bounds, SECTIONS)) + "\n", encoding="utf-8")
    meta = [";FFMETADATA1"]
    for (a, b), r in zip(bounds, SECTIONS):
        meta += ["[CHAPTER]", "TIMEBASE=1/1000", f"START={int(round(a * 1000))}", f"END={int(round(b * 1000))}",
                 f"title={ffmeta_escape(r['chapter_title'])}"]
    meta_path = PROJECT / "chapitres.ffmetadata"
    meta_txt = "\n".join(meta) + "\n"
    chapters_changed = not meta_path.exists() or meta_path.read_text(encoding="utf-8") != meta_txt
    meta_path.write_text(meta_txt, encoding="utf-8")

    # --- un fichier par section ------------------------------------------------------------
    if CUT_SECTIONS:
        VIDEO_DIR.mkdir(exist_ok=True)
        manifest_path = VIDEO_DIR / "manifest.json"          # bornes de chaque fichier : recoupe si le plan change
        manifest = load_json(manifest_path) if manifest_path.exists() else {}
        names = [f"{r['id']:02d}_{slugify(r['chapter_title'], 50)}{out_ext}" for r in SECTIONS]
        stale = [p for p in VIDEO_DIR.iterdir() if p.is_file() and re.match(r"\d{2}_", p.name) and p.name not in names]
        for p in stale:
            p.unlink()
            manifest.pop(p.name, None)
        if stale:
            print(f"{len(stale)} fichier(s) d'un plan précédent supprimé(s).")
        for (a, b), r, name in tqdm(list(zip(bounds, SECTIONS, names)), desc="Sections vidéo"):
            out = VIDEO_DIR / name
            key = [round(a, 3), round(b, 3), CUT_MODE]
            if out.exists() and manifest.get(name) == key and "video" not in FORCE:
                continue
            cmd = ["ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-ss", f"{a:.3f}", "-i", SOURCE, "-t", f"{b - a:.3f}",
                   "-map", "0:v:0", "-map", "0:a:0?"]
            if CUT_MODE == "copy":
                cmd += ["-c", "copy", "-avoid_negative_ts", "make_zero"]
            else:
                cmd += ["-c:v", "libx264", "-preset", "veryfast", "-crf", "20", "-pix_fmt", "yuv420p",
                        "-c:a", "aac", "-b:a", "160k", "-movflags", "+faststart"]
            run(cmd + [out.name], cwd=VIDEO_DIR)
            manifest[name] = key
            save_json(manifest_path, manifest)

    # --- vidéo complète avec chapitres intégrés -------------------------------------------
    if EMBED_CHAPTERS:
        chap_ext = SOURCE.suffix.lower() if keep_ext else ".mkv"
        out = PROJECT / f"{slugify(SOURCE.stem)}_chapitres{chap_ext}"
        if not out.exists() or chapters_changed or "video" in FORCE:
            run(["ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-i", SOURCE, "-i", PROJECT / "chapitres.ffmetadata",
                 "-map", "0:v:0", "-map", "0:a:0?", "-map_metadata", "1", "-map_chapters", "1", "-c", "copy", out])
        print("Vidéo avec chapitres :", out.name)

    print("\nChapitres YouTube :")
    print((PROJECT / "chapitres_youtube.txt").read_text(encoding="utf-8"))
    if CUT_SECTIONS:
        print("Fichiers :", *sorted(p.name for p in VIDEO_DIR.iterdir()), sep="\n  ")

Sections vidéo:   0%|          | 0/11 [00:00<?, ?it/s]

Vidéo avec chapitres : v-18dl8rj9s_chapitres.mp4

Chapitres YouTube :
0:00 Appels de Dieu dans Romains
9:20 Appartenir à Jésus
29:17 Abandon et paix intérieure
37:53 Limites du leadership spirituel
43:51 Harmonie spirituelle
57:02 Les différents appels à la foi
1:06:39 Appels spécifiques au service
1:12:55 Appel au Service Chrétien
1:19:24 Vies normales des apôtres
1:24:29 Appels spécifiques de Dieu
1:35:28 Appel distinct et vie personnelle

Fichiers :
  01_appels-de-dieu-dans-romains.mp4
  02_appartenir-a-jesus.mp4
  03_abandon-et-paix-interieure.mp4
  04_limites-du-leadership-spirituel.mp4
  05_harmonie-spirituelle.mp4
  06_les-differents-appels-a-la-foi.mp4
  07_appels-specifiques-au-service.mp4
  08_appel-au-service-chretien.mp4
  09_vies-normales-des-apotres.mp4
  10_appels-specifiques-de-dieu.mp4
  11_appel-distinct-et-vie-personnelle.mp4
  manifest.json


In [36]:
print("Résultats dans :", PROJECT)
for p in sorted(PROJECT.iterdir()):
    print("  ", p.name + ("/" if p.is_dir() else ""))

# Colab sans Drive : récupérer l'essentiel en un zip
if IN_COLAB and not MOUNT_DRIVE:
    from google.colab import files
    keep = [p for p in PROJECT.iterdir() if p.suffix in (".txt", ".json", ".md", ".ffmetadata")]
    tmp = PROJECT / "resultats"
    tmp.mkdir(exist_ok=True)
    for p in keep:
        shutil.copy(p, tmp / p.name)
    files.download(shutil.make_archive(str(PROJECT / "resultats"), "zip", tmp))

Résultats dans : /home/guillias/Bureau/socialMediaContents/outputs/v-18dl8rj9s
   audio.wav
   chapitres.ffmetadata
   chapitres_youtube.txt
   plan_thematique.txt
   sections.json
   sections_video/
   transcript.json
   transcription_structuree.md
   v-18dl8rj9s_chapitres.mp4


## Réglages et dépannage

| Constat | Réglage |
|---|---|
| Sections trop courtes / trop nombreuses | Augmente `TARGET_SECTION_SEC` (ex. 600), `MIN_SECTION_SEC` ou `TOPIC_STRICTNESS` (ex. 0.5), puis `FORCE = {"themes"}`. |
| Une notion longue est coupée | Garde `MERGE_CONTINUING = True` (le LLM fusionne les sections qui se suivent sur le même thème, jusqu'à `MAX_SECTION_SEC`) ; augmente `MAX_SECTION_SEC`. |
| Thèmes encore peu clairs | Utilise un LLM plus fort : `LLM_HF_MODEL = "Qwen/Qwen2.5-14B-Instruct"` (GPU), un modèle plus gros dans Ollama, ou le backend `anthropic`. Renseigne `GLOSSARY`. |
| Noms propres mal transcrits | Renseigne `INITIAL_PROMPT` puis `FORCE = {"transcribe"}`. |
| Colab : erreur `libcublas` / `libcudnn` | Le notebook tente de les charger et de les installer ; sinon il bascule sur CPU. Redémarre l'environnement et relance si besoin. |
| Colab : mémoire GPU insuffisante | Whisper puis le LLM sont chargés l'un après l'autre ; réduis le modèle (`LLM_HF_MODEL = "Qwen/Qwen2.5-3B-Instruct"`). |
| Aucun LLM disponible en local | Installe Ollama et fais `ollama pull qwen2.5:7b-instruct`, ou utilise le résumé extractif (thème = phrase la plus représentative). |
| Coupes vidéo décalées de quelques secondes | Utilise `CUT_MODE = "reencode"` (coupe exacte). |

## Limites

- Les frontières sont des **changements de sujet détectés automatiquement** (précision de l'ordre de la pause la plus proche) : relis le plan, puis ajuste les paramètres.
- Un LLM de 7 milliards de paramètres peut reformuler imparfaitement ou rater une fusion ; la relecture reste nécessaire.
- La qualité dépend de la transcription (bruit, accents, mélange de langues).